## Hands-on Exercise 1
### Deep Learning
#### Hamed Ahmadinia

In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.9.1+cu129
CUDA available: True
GPU: Tesla V100-SXM2-32GB


#### STEP 1 — Load MNIST

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transform
transform = transforms.Compose([
    transforms.ToTensor()
])

# Datasets
train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
test_dataset  = datasets.MNIST(root="./data", train=False, transform=transform, download=True)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

100%|██████████| 9.91M/9.91M [00:01<00:00, 9.53MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 253kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.38MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 19.7MB/s]

Train size: 60000
Test size: 10000


#### STEP 2 — Create Model Builder (Flexible Depth)

In [3]:
def build_mlp(input_size=784, hidden_size=128, depth=3, num_classes=10):
    layers = []
    in_features = input_size
    
    for _ in range(depth):
        layers.append(nn.Linear(in_features, hidden_size))
        layers.append(nn.ReLU())
        in_features = hidden_size
    
    layers.append(nn.Linear(hidden_size, num_classes))
    
    return nn.Sequential(*layers).to(device)

#### STEP 3 — Training Function

In [4]:
def train_model(depth, epochs=5):
    model = build_mlp(depth=depth)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for images, labels in train_loader:
            images = images.view(images.size(0), -1).to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"Depth {depth} | Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")
    
    return model

#### STEP 4 — Evaluation Function

In [5]:
def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.view(images.size(0), -1).to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return 100 * correct / total

#### STEP 5 — Run 3 Models

In [6]:
model_3 = train_model(depth=3)
acc_3 = evaluate(model_3)
print("Test Accuracy (3 layers):", acc_3)

Depth 3 | Epoch 1/5 | Loss: 0.3304
Depth 3 | Epoch 2/5 | Loss: 0.1269
Depth 3 | Epoch 3/5 | Loss: 0.0902
Depth 3 | Epoch 4/5 | Loss: 0.0682
Depth 3 | Epoch 5/5 | Loss: 0.0548
Test Accuracy (3 layers): 97.69


In [7]:
model_5 = train_model(depth=5)
acc_5 = evaluate(model_5)
print("Test Accuracy (5 layers):", acc_5)

Depth 5 | Epoch 1/5 | Loss: 0.3778
Depth 5 | Epoch 2/5 | Loss: 0.1318
Depth 5 | Epoch 3/5 | Loss: 0.0932
Depth 5 | Epoch 4/5 | Loss: 0.0757
Depth 5 | Epoch 5/5 | Loss: 0.0616
Test Accuracy (5 layers): 97.19


In [8]:
model_10 = train_model(depth=10)
acc_10 = evaluate(model_10)
print("Test Accuracy (10 layers):", acc_10)

Depth 10 | Epoch 1/5 | Loss: 0.5796
Depth 10 | Epoch 2/5 | Loss: 0.2055
Depth 10 | Epoch 3/5 | Loss: 0.1591
Depth 10 | Epoch 4/5 | Loss: 0.1254
Depth 10 | Epoch 5/5 | Loss: 0.1034
Test Accuracy (10 layers): 96.86


## Results Summary

| Depth      | Final Loss | Test Accuracy (%) |
|------------|------------|-------------------|
| 3 layers   | 0.0548     | 97.69%            |
| 5 layers   | 0.0616     | 97.19%            |
| 10 layers  | 0.1034     | 96.86%            |

##### Increasing network depth from 3 to 10 hidden layers did not improve classification accuracy on MNIST.
##### In fact, performance slightly degraded. 
##### This suggests that the MNIST dataset does not require deep hierarchical feature representations, and that deeper architectures introduce optimization difficulties without providing additional representational benefit. 
##### The results highlight that model complexity must align with dataset complexity.

## Training Accuracy 

In [9]:
def evaluate_train(model):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in train_loader:
            images = images.view(images.size(0), -1).to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return 100 * correct / total

In [10]:
print("Train acc (3):", evaluate_train(model_3))
print("Train acc (5):", evaluate_train(model_5))
print("Train acc (10):", evaluate_train(model_10))

Train acc (3): 98.98666666666666
Train acc (5): 98.37333333333333
Train acc (10): 97.86


## Model Comparison Results

| Depth | Train Accuracy (%) | Test Accuracy (%) |
|-------|--------------------|-------------------|
| 3     | 98.99              | 97.69             |
| 5     | 98.37              | 97.19             |
| 10    | 97.86              | 96.86             |

##### Increasing the number of hidden layers from 3 to 10 did not improve classification accuracy on MNIST.
##### In fact, the 3-layer model achieved the highest test accuracy (97.69%), while deeper models slightly underperformed.
##### This suggests that MNIST does not require very deep representations and that increasing depth introduces optimization challenges without additional representational benefit.
##### The results demonstrate that model complexity should align with dataset complexity.